# Getting started: Spark on the Lichess lake

This kernel is a Spark driver. In the cluster, executors start on the spot batch nodes when a query needs them
(the first one waits ~3–5 min for a node) and are removed after 5 idle minutes. On your PC the same code runs in local mode.

Spark UI: `kubectl port-forward -n lichess svc/jupyter 8888 4040`, then http://localhost:4040

In [ ]:
from pyspark.sql import functions as F
from lichess_spark import dest_root, notebook_spark

spark = notebook_spark()
root = dest_root()
spark

## Raw games

One Parquet dataset per month under `raw/games`, every PGN header tag as a string column plus `movetext`.
Until the Iceberg catalog exists (#9, #23) we read the files directly.

In [ ]:
games = spark.read.option("recursiveFileLookup", "true").parquet(f"{root}/raw/games")
games.printSchema()

In [ ]:
per_month = games.groupBy("month").count().orderBy("month").toPandas()
per_month.plot(x="month", y="count", kind="bar", legend=False, title="Games per month", figsize=(10, 3));

## White's score by rating

Score = 1 for a win, ½ for a draw. Rated games with both ratings known.

In [ ]:
scored = (games
    .where(F.col("Event").startswith("Rated") & F.col("WhiteElo").rlike("^[0-9]+$") & F.col("BlackElo").rlike("^[0-9]+$"))
    .withColumn("bucket", (F.col("WhiteElo").cast("int") / 100).cast("int") * 100)
    .withColumn("score", F.when(F.col("Result") == "1-0", 1.0).when(F.col("Result") == "1/2-1/2", 0.5).otherwise(0.0)))

by_rating = (scored.groupBy("bucket")
    .agg(F.count(F.lit(1)).alias("games"), F.avg("score").alias("white_score"))
    .where("games >= 1000").orderBy("bucket").toPandas())
by_rating.plot(x="bucket", y="white_score", title="White's score by White's rating", figsize=(10, 3));

## When you're done

`spark.stop()` releases the executors right away (they'd go after 5 idle minutes anyway).
JupyterLab itself stops after the workflow's `hours`; delete the workflow to stop it sooner.
Files in this server are on a disk that's deleted with the cluster, so commit notebooks you want to keep.

In [ ]:
spark.stop()